# 02 — Feature Engineering

**Purpose:** Load raw M1 data and compute the full feature set across 5 groups:
1. Technical indicators (EMA, RSI, MACD, ADX, Bollinger, z-scores)
2. Volatility (ATR, realised vol, Parkinson, Garman-Klass)
3. Candle structure (body, wicks, range, direction)
4. Session / calendar time features
5. Cross-asset features (returns, correlations, lead-lag)

**No future data is used.** All features are computed using only past bars.

**Inputs:** `data/raw/`  
**Outputs:** `data/features/{symbol}_features.parquet`

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.utils.config import load_config, get_symbol, get_aux_symbols, get_paths, get_feature_params
from src.data.mt5_loader import MT5Loader
from src.features.technical import compute_all_technical
from src.features.volatility import compute_all_volatility
from src.features.time_features import compute_all_time_features
from src.features.cross_asset import compute_all_cross_asset

pd.set_option("display.max_columns", 40)

cfg     = load_config()
paths   = get_paths(cfg, "..")
symbol  = get_symbol(cfg)
feat_cfg = get_feature_params(cfg)

print(f"Symbol: {symbol}")
print(f"Feature params:\n{feat_cfg}")

## 1. Load raw data

In [ ]:
loader = MT5Loader(cfg)
df = loader.load_raw(symbol, "M1")

print(f"Loaded {len(df)} M1 bars: {df.index[0]} → {df.index[-1]}")
display(df.head())

In [ ]:
# Load auxiliary symbols
aux_symbols = get_aux_symbols(cfg, enabled_only=True)
aux_dict = {}
for sym in aux_symbols:
    try:
        aux_dict[sym] = loader.load_raw(sym, "M1")
        print(f"  Loaded {sym}: {len(aux_dict[sym])} bars")
    except FileNotFoundError:
        print(f"  {sym}: not found — skipping (run notebook 01 first)")

## 2. Technical features

In [ ]:
tech_features = compute_all_technical(df, feat_cfg)
print(f"Technical features: {tech_features.shape[1]} columns")
display(tech_features.tail(3))

## 3. Volatility features

In [ ]:
vol_features = compute_all_volatility(df, feat_cfg)
print(f"Volatility features: {vol_features.shape[1]} columns")
display(vol_features.tail(3))

## 4. Time / session features

In [ ]:
time_features = compute_all_time_features(df, session_start_hour=7)
print(f"Time features: {time_features.shape[1]} columns")
display(time_features.head(3))

## 5. Cross-asset features

In [ ]:
cross_features = compute_all_cross_asset(df, aux_dict, feat_cfg)
print(f"Cross-asset features: {cross_features.shape[1]} columns")
if not cross_features.empty:
    display(cross_features.tail(3))

## 6. Combine and save

In [ ]:
# Combine all feature groups with original OHLCV
all_features = pd.concat([df, tech_features, vol_features, time_features, cross_features], axis=1)

# Drop the first N rows where rolling features are NaN (warmup period)
# Keep only rows after the longest warmup (EMA 200)
warmup = max(feat_cfg.get("ema_periods", [200]))
all_features = all_features.iloc[warmup:].copy()

print(f"Final feature matrix: {all_features.shape}")
print(f"NaN count per column (top 10):")
nan_counts = all_features.isnull().sum().sort_values(ascending=False)
print(nan_counts.head(10))

# Save
feat_path = paths["features"] / f"{symbol}_features.parquet"
feat_path.parent.mkdir(parents=True, exist_ok=True)
all_features.to_parquet(feat_path, engine="pyarrow", compression="snappy")
print(f"\nSaved → {feat_path}")

## 7. Exploratory analysis

In [ ]:
# Intraday volatility heatmap (ATR by hour and day-of-week)
df_plot = all_features[["atr_14", "hour", "day_of_week"]].dropna()
pivot = df_plot.groupby(["day_of_week", "hour"])["atr_14"].mean().unstack("hour")

days = ["Mon", "Tue", "Wed", "Thu", "Fri"]
pivot.index = [days[i] for i in pivot.index if i < len(days)]

plt.figure(figsize=(16, 4))
sns.heatmap(pivot, cmap="YlOrRd", linewidths=0.5, fmt=".2f", annot=False)
plt.title("Average ATR by Hour (UTC) and Day of Week")
plt.xlabel("Hour (UTC)")
plt.ylabel("Day")
plt.tight_layout()
plt.savefig("../reports/atr_heatmap.png", dpi=120)
plt.show()

In [ ]:
# Feature correlation heatmap (selected key features)
key_cols = [
    "log_ret_1", "rsi_14", "macd_hist", "adx", "bb_pct_b",
    "atr_14", "vol_20", "vol_ratio_20_120",
    "session_xetra_open", "session_ny_open",
]
key_cols = [c for c in key_cols if c in all_features.columns]

corr = all_features[key_cols].dropna().corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.savefig("../reports/feature_correlation.png", dpi=120)
plt.show()